# Start a local MCP server using the hana_ai MCP client for the SAP HANA Cloud Object Discovery- and Data Retrieval-Tools
- It showcase to use an MCP client over the tools (`hana_ai.tools.hana_ml_tools.graph_tools`)
    - `ObjectDiscoveryTool` → `SYS.AI_OBJECT_DISCOVERY`
    - `DataRetrievalTool`   → `SYS.AI_DATA_RETRIEVAL`
- The goal of this notebook is to see the audit projection for each tool-call cleanly, without agent orchestration noise
<br>

- Including **MCP client audit mode** via HANA session Variables
    - If only *session-variable* `audit_enabled=True` is but `audit_log_path=None` and no `MCP_AUDIT_LOG_PATH` env var is set, the toolkit does not write any local JSONL sink;
    - However the hana-ai toolkit MCP client projects the session variable back into the **shared HANA session** via `SET '<KEY>' = '<VALUE>'`, readable back with `SESSION_CONTEXT('<KEY>')`. The same session variable audit informataion can be queried on the SAP HANA Cloud database (M_SESSION_CONTEXT, `SESSION_CONTEXT(...)`)
    - Moreover additional SAP HANA Cloud server-side audit trail policies can be enabled and applied (`AUDIT_LOG` + `CREATE AUDIT POLICY`).
<br>

- The ObjectDiscoveryTool + DataRetrievalTool are wrapped into a `hana_ai.tools.toolkit.**HANAMLToolkit**' instance
    - this tool-objects are exposed based on a HANA Cloud connection and db-user
    - the **HANAMLToolkit** object allows to __launch_mcp_server__
<br>

- The **build_context_agent_mcp_tools** from hana_ai.tools.hana_ml_tools.utility now allow to initialize a MCP client
    - The MCP client tools are called without an additional / client or DB user authentication
    - The database connection of the respective tool added to the MCP Server is used against the HANA instance for each user of this MCP client instance
    - Some session variables can be specified, however key session variables cannot be changed


<br><br>

# Loading packages and Connecting to SAP HANA Cloud

The following HANA Cloud connection
- A technical HANA Cloud user, with ~ ObjectDiscovery/DataRetrieval-Tool User (not developer) privileges, example HANA_MCP_TU

Prerequisites
- packages installed: hana_ml, hana_ai from [pypi.org/project/hana-ai/](https://pypi.org/project/hana-ai/), documentation [sap.github.io/generative-ai-toolkit-for-sap-hana-cloud/](https://sap.github.io/generative-ai-toolkit-for-sap-hana-cloud/), repo [github.com/SAP/generative-ai-toolkit-for-sap-hana-cloud](https://github.com/SAP/generative-ai-toolkit-for-sap-hana-cloud)
- hana_ai requirements incl.: `langchain-community, langchain-text-splitters, numpy, pandas, hana-ml>=2.27.26020601`, `pydantic>=2.11,<2.13`, `sap-ai-sdk-gen[all], langchain-hana, fastmcp>=2.0,<3.0`

In [1]:
# Show python environment used
import sys
print(sys.base_prefix)
print(sys.prefix)

C:\Users\D059078\AppData\Local\Programs\Python\Python312
C:\Users\D059078\AppData\Local\Programs\Python\envs\agaienv


In [2]:
# Minimum version of hana_ai: 1.1.26071300
import hana_ml
import hana_ai
print(hana_ai.__version__)

1.1.26071300


<br><br>

# Define hana_ai-HANAMLToolkit tools and start a local MCP server
## 1 Connect to SAP HANA Cloud

Connect to HANA Cloud using a __Technical User (TU)__ with __Object Discovery__ and __Data Retrieval__-tool usage privileges

In [3]:
# Connect to HANA Cloud using a Technical User (TU) with Tool usage privileges
from hana_ml import ConnectionContext
from hana_ml import dataframe
from hana_ml.dataframe import create_dataframe_from_pandas

cc = ConnectionContext(userkey='HC34e77cb1_MCP_TU') # Using a shared / technical HANA database user id, certainly can also be a developer's private database user or other
print(cc.hana_version())

4.00.000.00.1785230653 (CE2026.28)


In [4]:
# Confirm which HANA session (and thus which SESSION_CONTEXT scope) we're reading from.
session_probe = cc.sql(
    "SELECT CURRENT_USER AS \"USER\", CURRENT_CONNECTION AS \"CONNECTION_ID\", SESSION_USER AS \"SESSION_USER\" FROM DUMMY"
).collect()
session_probe

,USER,CONNECTION_ID,SESSION_USER
0,HANA_MCP_TU,2122530,HANA_MCP_TU


<br><br>

## 2 Map Object Discovery & Data Retrieval tools to a hana_ai-HANAMLToolkit object

The toolkit is created with empty list of `used_tools=[]` and `audit_log_path=None` (no JSONL sink provided).
- Standard hana_ai-tools like `fetch_data`, or no hana_ml training/forecast tools,
- The two retrieval tools are added as custom tools, both pointing at the same AI Core remote source and the same SYSTEM-side metadata schema/prefix. The RAG schema/table/graph names are auto-derived from the prefix and schema.

In [5]:
# Based on the TU-connection, map the ObjectDiscovery- and DataRetrieval-Tools into a HANAMLToolkit
# Create toolkit
from hana_ai.tools.toolkit import HANAMLToolkit
from hana_ai.tools.hana_ml_tools.graph_tools import ObjectDiscoveryTool, DataRetrievalTool

# cc = TU-connection
toolkit = HANAMLToolkit(connection_context=cc)

toolkit = HANAMLToolkit(
        connection_context=cc,
        used_tools=[],        # start empty and add exactly the two retrieval tools below
    audit_enabled=True,       # client-local session auditing enabled
    audit_log_path=None,      # session-variable-only audit: no JSONL sink
    )

In [6]:
# Adding "Wrapper Procedure"-objects around the raw ObjectDiscovery- and DataRetrieval-Tools to the toolkit
DISCOVERY_PROCEDURE_NAME='EPM_OBJECTDISCOVERY_TOOL95'
RETRIEVAL_SCHEMA_NAME='BTP4AI_95'
DATA_RETRIEVAL_PROCEDURE_NAME='EPM_DATARETRIEVAL_TOOL95'

#discovery_tool = ObjectRetrievalTool(connection_context=cc)
discovery_tool = ObjectDiscoveryTool(connection_context=cc)
discovery_tool.configure(
        schema_name=RETRIEVAL_SCHEMA_NAME,
        procedure_name=DISCOVERY_PROCEDURE_NAME,
    )
discovery_tool.description = 'tool for discovering data in the HANA Cloud system. Ask natural language questions, get a database schema description.'
discovery_tool.name = 'objectdiscovery_tool'
toolkit.add_custom_tool(discovery_tool)
 
data_tool = DataRetrievalTool(connection_context=cc)
data_tool.configure(
        schema_name=RETRIEVAL_SCHEMA_NAME,
        procedure_name=DATA_RETRIEVAL_PROCEDURE_NAME,
    )
data_tool.description = 'tool for retrieve data from a HANA Cloud system. Ask natural language questions or SQL, get a response based on data.'
data_tool.name = 'data_retrieval_tool'
toolkit.add_custom_tool(data_tool)

<br><br>

## 3 For the HANAMLToolkit object, launch local MCP instance

The tools are DIRECTLY mapped from the toolkit to an MCP server using the `<toolkit>.launch_mcp_server`-method.
- the tools are NOT routed through and additional hana_ai-`ContextAgent` (ReAct-Agent).

A local FastMCP server is started advertising the tools.

In [7]:
from hana_ai.tools.hana_ml_tools.utility import find_free_port

In [8]:
MCP_HOST = "127.0.0.1"
MCP_PORT = find_free_port()
MCP_BASE_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

In [9]:
# Note toolkit object from class HANAMLToolkit is bound to a HANA Connection Context set earlier: cc
toolkit.launch_mcp_server(
    transport="http",
    host=MCP_HOST,
    port=MCP_PORT,
    max_retries=5,
)

import time
time.sleep(1.5)
print(MCP_BASE_URL)

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 2.14.5                                │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                    🖥  Server:      HANATools                                 │                  
                 │                    🚀 Deploy free: https://fastmcp.cloud                     │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯                  
                 ╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                          ✨ FastMCP 3.0 is coming!                           │                  
                 │       Pin `fastmcp < 3` in production, then upgrade when you're ready.       │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[08/06/26 14:01:07] INFO     Starting MCP server 'HANATools' with transport 'http' on                ]8;id=166694;file://C:\Users\D059078\AppData\Local\Programs\Python\envs\agaienv\Lib\site-packages\fastmcp\server\server.py\server.py]8;;\:]8;id=686675;file://C:\Users\D059078\AppData\Local\Programs\Python\envs\agaienv\Lib\site-packages\fastmcp\server\server.py#2580\2580]8;;\
                             http://127.0.0.1:8600/mcp                                                             

INFO:     Started server process [55008]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8600 (Press CTRL+C to quit)


http://127.0.0.1:8600/mcp


<br><br>

## Cleanup

In [ ]:
if mcp_client is not None:
    try:
        mcp_client.close()
    except Exception:
        pass

if toolkit is not None:
    try:
        toolkit.stop_mcp_server(host=MCP_HOST, port=MCP_PORT, transport="http", force=True, timeout=3.0)
    except Exception:
        pass

# Close the DBA-style observer session opened for M_SESSION_CONTEXT cross-session read.
try:
    observer_cc.connection.close()
except Exception:
    pass

try:
    cc.connection.close()
except Exception:
    pass

print("Cleanup finished.")

---